In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import torch
from livelossplot import PlotLosses
from torch import nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import MNIST

import decent_bench.utils.interoperability as iop
from decent_bench import benchmark
from decent_bench.agents import Agent
from decent_bench.algorithms import decentralized
from decent_bench.algorithms.utils import pytorch_initialization
from decent_bench.costs import PyTorchCost
from decent_bench.datasets import PyTorchDatasetHandler
from decent_bench.metrics import metric_library as ml
from decent_bench.networks import P2PNetwork
from decent_bench.utils.pytorch_utils import ArgmaxActivation, SimpleLinearModel, visualize_scheduler
from decent_bench.utils.types import SupportedDevices, SupportedFrameworks
from examples.nim.src.algorithms.lt_admm_ema import LT_ADMM_EMA

In [ ]:
iterations = 1000
n_trials = 1
n_agents = 5
state_snapshot_period = 50
samples_per_partition = 1000
batch_size = 64
local_steps = 10
step_size = 0.01
device = SupportedDevices.CPU
use_dataloader = False
compile_model = False
opt_cls = torch.optim.Adam
opt_kwargs = None
sched_cls = torch.optim.lr_scheduler.StepLR
sched_kwargs = {"step_size": 100, "gamma": 0.9}

table_metrics = [
    ml.ConsensusError([min, np.average, max]),
    ml.GradientCalls([np.average, sum]),
    ml.SentMessages([np.average, sum]),
    ml.Accuracy([min, np.average, max], fmt=".2%"),
    ml.Precision([min, np.average, max], fmt=".2%"),
    ml.Recall([min, np.average, max], fmt=".2%"),
    ml.Loss([min, np.average, max]),
]

plot_metrics = [
    ml.ConsensusError([], x_log=False, y_log=True),
    ml.Accuracy([], x_log=False, y_log=False),
    ml.Precision([], x_log=False, y_log=False),
    ml.Recall([], x_log=False, y_log=False),
    ml.Loss([], x_log=False, y_log=False),
]

In [ ]:
torch_device = iop.device_to_framework_device(device, SupportedFrameworks.PYTORCH)
transform = transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: x.view(-1))])
mnist_train = MNIST(root="data", train=True, download=True, transform=transform, target_transform=torch.tensor)
mnist_test = MNIST(root="data", train=False, download=True, transform=transform, target_transform=torch.tensor)

train_dataset = PyTorchDatasetHandler(
    torch_dataset=mnist_train,
    n_features=28 * 28,
    n_targets=10,
    n_partitions=n_agents,
    samples_per_partition=samples_per_partition,
    heterogeneity=False,
)
# train_dataset = PyTorchDatasetHandler(
#     torch_dataset=mnist_train,
#     n_features=28 * 28,
#     n_targets=10,
#     n_partitions=n_agents,
#     samples_per_partition=samples_per_partition,
#     heterogeneity=True,
#     targets_per_partition=2,
# )
test_dataset = PyTorchDatasetHandler(
    torch_dataset=mnist_test,
    n_features=28 * 28,
    n_targets=10,
    n_partitions=n_agents,
    samples_per_partition=samples_per_partition,
    heterogeneity=False,
)
# test_dataset = PyTorchDatasetHandler(
#     torch_dataset=mnist_test,
#     n_features=28 * 28,
#     n_targets=10,
#     n_partitions=n_agents,
#     samples_per_partition=samples_per_partition,
#     heterogeneity=True,
#     targets_per_partition=2,
# )


def model_generator() -> torch.nn.Module:
    """Generate a simple linear model for the MNIST dataset."""
    return SimpleLinearModel(
        input_size=train_dataset.n_features,
        hidden_sizes=[32, 16],
        output_size=train_dataset.n_targets,
    )

### Setup benchmark problem

In [ ]:
costs = [
    PyTorchCost(
        dataset=p,
        model=model_generator(),
        loss_fn=nn.CrossEntropyLoss(),
        final_activation=ArgmaxActivation(),
        batch_size=batch_size,
        device=device,
        use_dataloader=use_dataloader,
        dataloader_kwargs={} if device == SupportedDevices.GPU else {"pin_memory": True},
        compile_model=compile_model,
        compile_kwargs={"mode": "reduce-overhead"},
    )
    for p in train_dataset.get_partitions()
]
agents = [Agent(i, cost, state_snapshot_period=state_snapshot_period) for i, cost in enumerate(costs)]
graph = nx.complete_graph(n_agents)
network = P2PNetwork(graph=graph, agents=agents)
problem = benchmark.BenchmarkProblem(network=network, test_data=test_dataset.get_datapoints())
x0 = pytorch_initialization(network)
algorithms = [
    decentralized.DiNNO(
        iterations=iterations,
        x0=x0,
        step_size=step_size,
        local_steps=local_steps,
    ),
    decentralized.LT_ADMM(
        iterations=iterations,
        x0=x0,
        step_size=step_size,
        local_steps=local_steps,
    ),
    LT_ADMM_EMA(
        iterations=iterations,
        x0=x0,
        step_size=step_size,
        local_steps=local_steps,
        send_ema_x=False,
        name="LT-ADMM-EMA-Optim-F",
    ),
    LT_ADMM_EMA(
        iterations=iterations,
        x0=x0,
        step_size=step_size,
        local_steps=local_steps,
        opt_cls=opt_cls,
        opt_kwargs=opt_kwargs,
        sched_cls=sched_cls,
        sched_kwargs=sched_kwargs,
        send_ema_x=False,
        name="LT-ADMM-EMA-Torch-F",
    ),
    LT_ADMM_EMA(
        iterations=iterations,
        x0=x0,
        step_size=step_size,
        local_steps=local_steps,
        send_ema_x=True,
        name="LT-ADMM-EMA-Optim-T",
    ),
    LT_ADMM_EMA(
        iterations=iterations,
        x0=x0,
        step_size=step_size,
        local_steps=local_steps,
        opt_cls=opt_cls,
        opt_kwargs=opt_kwargs,
        sched_cls=sched_cls,
        sched_kwargs=sched_kwargs,
        send_ema_x=True,
        name="LT-ADMM-EMA-Torch-T",
    ),
]

In [ ]:
# Simple visualization of the learning rate scheduler
test_scheduler = sched_cls(opt_cls(model_generator().parameters(), lr=step_size), **sched_kwargs)
visualize_scheduler(test_scheduler, iterations)

### Run benchmark

In [ ]:
result = benchmark.benchmark(algorithms=algorithms, benchmark_problem=problem, n_trials=n_trials, show_speed=True)

### Compute metrics

In [ ]:
metric_result = benchmark.compute_metrics(
    benchmark_result=result,
    table_metrics=table_metrics,
    plot_metrics=plot_metrics,
)

### Display metrics

In [ ]:
benchmark.display_metrics(metrics_result=metric_result)

### Visualize agents

In [ ]:
test_loader = DataLoader(test_dataset.get_datapoints(), batch_size=10, shuffle=True)
data_iter = iter(test_loader)
images, labels = next(data_iter)
images, labels = images.to(torch_device), labels.to(torch_device)
for alg, networks in result.states.items():
    display(f"Results for {alg.name}, {len(networks)} trials")
    for n in networks:
        all_agents = n.agents()
        all_preds = [[] for _ in range(len(labels))]
        for agent in all_agents:
            if not isinstance(agent.cost, PyTorchCost):
                raise ValueError("Cost is not a PyTorchCost, cannot compute predictions.")
            preds = agent.cost.predict(agent.x, images)
            for i in range(len(labels)):
                all_preds[i].append(preds[i])

        fig, ax = plt.subplots(2, len(labels) // 2, figsize=(17, 7))
        for i in range(len(labels)):
            row = i // (len(labels) // 2)
            col = i % (len(labels) // 2)
            ax[row, col].imshow(images[i].reshape(28, 28).cpu(), cmap="gray")
            ax[row, col].set_title(f"Guess for Agents: {', '.join(map(str, all_preds[i]))}")
            ax[row, col].axis("off")
        plt.show()

### Centralized solution

In [ ]:
model = model_generator()
model.to(torch_device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 15
criterion = nn.CrossEntropyLoss()
liveloss = PlotLosses()
model.train()
dataloader = DataLoader(train_dataset.get_datapoints(), batch_size=batch_size, shuffle=True, pin_memory=True)
for epoch in range(epochs):
    running_loss = 0.0
    for inputs, targets in dataloader:
        inputs, targets = inputs.to(torch_device), targets.to(torch_device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.detach()
    epoch_loss = running_loss / len(dataloader)
    liveloss.update({"loss": epoch_loss.item()})
    liveloss.send()

In [ ]:
# Visualize a few mnist test samples with predictions
model.eval()
outputs = model(images)
predicted = torch.argmax(outputs, 1)
# Plot images with predicted labels
fig, axes = plt.subplots(2, 5, figsize=(12, 6))
for i in range(10):
    ax = axes[i // 5, i % 5]
    ax.imshow(images[i].cpu().view(28, 28), cmap="gray")
    ax.set_title(f"Predicted: {predicted[i].item()}")
    ax.axis("off")
plt.show()